# NIBRS Generalization Testing v2 — Pipeline 3

**Purpose:** Test how well Chicago-trained v2 models generalize to Illinois 2024 NIBRS data.

**Pipeline flow:**
```
NIBRS Raw Data  →  30 Feature Engineering  →  Load saved artifacts  →  Predict  →  Evaluate
```

**Key rules (prevent leakage / ensure fair generalization test):**

| Artifact | Action |
|---|---|
| `data_scaler_v2.joblib` | **LOAD & TRANSFORM only** — never refit on NIBRS |
| `composite_weights.joblib` | **LOAD & APPLY only** — never recompute on NIBRS |
| `feature_names_v2.joblib` | **LOAD** — align NIBRS features to exact same 30-feature order |
| `crime_models_v2.joblib` | **LOAD & PREDICT** — no retraining |

**Improvements over original NIBRS testing notebook:**

| Area | Original | v2 |
|---|---|---|
| Features | 12 (raw sums) | **30** (means, surge, trend, cross-tier, composite) |
| Lag type | Raw sum | **Rolling mean** (scale-independent) |
| Models | Phase 2 baseline | **v2 models** (calibrated XGBoost, tuned weights) |
| Composite risk | None | **Loaded from Chicago weights** |


## Step 1: Imports & Load Chicago Artifacts

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# ── CalibratedXGB must be defined before loading models ────────────────────
class CalibratedXGB:
    def __init__(self, xgb_model, iso_reg):
        self.xgb_model = xgb_model
        self.iso_reg   = iso_reg
    def predict_proba(self, X):
        raw = self.xgb_model.predict_proba(X)[:, 1]
        cal = np.clip(self.iso_reg.predict(raw), 0, 1)
        return np.column_stack([1-cal, cal])

# ── Paths ──────────────────────────────────────────────────────────────────
FOLDER    = 'G:/My Drive/To be - AI Medical Engineer/Claude_app_Project/IT5006 Group Project Final'
SAVE_DIR  = os.path.join(FOLDER, 'Saved_Models')
NIBRS_DIR = os.path.join(FOLDER, 'Testing Dataset NIBRS')

TARGET_COLS = ['y_tier1', 'y_tier2', 'y_tier3', 'y_tier4']
THRESHOLDS  = [0.10, 0.20, 0.35, 0.35]
NEEDS_SCALING = {
    'Logistic Regression': True,
    'Random Forest'      : False,
    'XGBoost'            : False,
    'Neural Network'     : True
}
TIERS = [
    'Tier 1 - Lethal',
    'Tier 2 - Personal Violence',
    'Tier 3 - Property',
    'Tier 4 - Public Order'
]

# ── Load Chicago-trained artifacts ─────────────────────────────────────────
# IMPORTANT: load only — never refit/recompute on NIBRS data
loaded_models     = joblib.load(os.path.join(SAVE_DIR, 'crime_models_v2.joblib'))
loaded_scaler     = joblib.load(os.path.join(SAVE_DIR, 'data_scaler_v2.joblib'))
FEATURES          = joblib.load(os.path.join(SAVE_DIR, 'feature_names_v2.joblib'))
composite_weights = joblib.load(os.path.join(SAVE_DIR, 'composite_weights.joblib'))

print('Chicago artifacts loaded:')
print(f'  Models    : {list(loaded_models.keys())}')
print(f'  Features  : {len(FEATURES)} features')
print(f'  Scaler    : fitted on Chicago 2015-2024 training data')
print(f'  Weights   : {list(composite_weights.keys())}')
print()
print('Feature list:')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i:2d}. {f}')

Chicago artifacts loaded:
  Models    : ['Logistic Regression', 'Random Forest', 'XGBoost', 'Neural Network']
  Features  : 30 features
  Scaler    : fitted on Chicago 2015-2024 training data
  Weights   : ['y_tier1', 'y_tier2', 'y_tier3', 'y_tier4']

Feature list:
   1. month_sin
   2. month_cos
   3. dow_sin
   4. dow_cos
   5. tier1_lag_7d_mean
   6. tier2_lag_7d_mean
   7. tier3_lag_7d_mean
   8. tier4_lag_7d_mean
   9. tier1_lag_30d_mean
  10. tier2_lag_30d_mean
  11. tier3_lag_30d_mean
  12. tier4_lag_30d_mean
  13. tier1_lag_90d_mean
  14. tier2_lag_90d_mean
  15. tier3_lag_90d_mean
  16. tier4_lag_90d_mean
  17. tier1_surge_ratio
  18. tier2_surge_ratio
  19. tier3_surge_ratio
  20. tier4_surge_ratio
  21. tier1_trend
  22. tier2_trend
  23. tier3_trend
  24. tier4_trend
  25. violence_property_ratio
  26. disorder_to_violence
  27. composite_risk_tier1
  28. composite_risk_tier2
  29. composite_risk_tier3
  30. composite_risk_tier4


## Step 2: NIBRS Offense Code → Severity Tier Mapping
NIBRS uses numeric offense codes (e.g. `09A` = Murder).
Map to the same 4-tier framework used in Chicago training.


In [2]:
NIBRS_SEVERITY_MAPPING = {
    # TIER 1 — LETHAL
    '09A': 'Tier 1 - Lethal',   # Murder / Non-negligent Manslaughter
    '09B': 'Tier 1 - Lethal',   # Negligent Manslaughter
    '11A': 'Tier 1 - Lethal',   # Rape
    '100': 'Tier 1 - Lethal',   # Kidnapping / Abduction
    '64A': 'Tier 1 - Lethal',   # Human Trafficking - Commercial Sex
    '64B': 'Tier 1 - Lethal',   # Human Trafficking - Involuntary Servitude

    # TIER 2 — PERSONAL VIOLENCE
    '120': 'Tier 2 - Personal Violence',  # Robbery
    '13A': 'Tier 2 - Personal Violence',  # Aggravated Assault
    '13B': 'Tier 2 - Personal Violence',  # Simple Assault
    '11B': 'Tier 2 - Personal Violence',  # Sodomy
    '11C': 'Tier 2 - Personal Violence',  # Sexual Assault With An Object
    '11D': 'Tier 2 - Personal Violence',  # Fondling
    '36A': 'Tier 2 - Personal Violence',  # Incest
    '36B': 'Tier 2 - Personal Violence',  # Statutory Rape

    # TIER 3 — PROPERTY
    '220': 'Tier 3 - Property',   # Burglary
    '23A': 'Tier 3 - Property',   # Pocket-picking
    '23C': 'Tier 3 - Property',   # Shoplifting
    '23H': 'Tier 3 - Property',   # All Other Larceny
    '240': 'Tier 3 - Property',   # Motor Vehicle Theft
    '200': 'Tier 3 - Property',   # Arson
    '290': 'Tier 3 - Property',   # Destruction of Property
    '13C': 'Tier 3 - Property',   # Intimidation
    '26A': 'Tier 3 - Property',   # False Pretenses / Fraud
    '90J': 'Tier 3 - Property',   # Trespass

    # TIER 4 — PUBLIC ORDER
    '35A': 'Tier 4 - Public Order',  # Drug / Narcotic Violations
    '35B': 'Tier 4 - Public Order',  # Drug Equipment Violations
    '40A': 'Tier 4 - Public Order',  # Prostitution
    '520': 'Tier 4 - Public Order',  # Weapon Law Violations
    '39A': 'Tier 4 - Public Order',  # Betting
    '90C': 'Tier 4 - Public Order',  # Disorderly Conduct
    '90D': 'Tier 4 - Public Order',  # Driving Under Influence
    '90G': 'Tier 4 - Public Order',  # Liquor Law Violations
}

print(f'Tier mapping defined: {len(NIBRS_SEVERITY_MAPPING)} offense codes mapped.')
for tier in TIERS:
    count = sum(1 for v in NIBRS_SEVERITY_MAPPING.values() if v == tier)
    print(f'  {tier}: {count} offense codes')


Tier mapping defined: 32 offense codes mapped.
  Tier 1 - Lethal: 6 offense codes
  Tier 2 - Personal Violence: 8 offense codes
  Tier 3 - Property: 10 offense codes
  Tier 4 - Public Order: 8 offense codes


## Step 3: Load & Merge NIBRS Data

In [3]:
print('Loading NIBRS files...')
offenses  = pd.read_csv(os.path.join(NIBRS_DIR, 'NIBRS_OFFENSE_Illinois.csv'))
incidents = pd.read_csv(
    os.path.join(NIBRS_DIR, 'NIBRS_incident_Illinois.csv'),
    low_memory=False
)

# Merge offense + incident to get agency_id and date
df_nibrs = offenses.merge(
    incidents[['incident_id', 'incident_date', 'agency_id']],
    on='incident_id', how='inner'
)

df_nibrs['incident_date'] = pd.to_datetime(df_nibrs['incident_date'], errors='coerce')
df_nibrs = df_nibrs.dropna(subset=['incident_date', 'agency_id', 'offense_code'])

# Map offense codes to severity tiers
df_nibrs['Severity_Tier'] = df_nibrs['offense_code'].map(NIBRS_SEVERITY_MAPPING).fillna('Other')

print(f'NIBRS merged shape  : {df_nibrs.shape}')
print(f'Date range          : {df_nibrs["incident_date"].min().date()} -> {df_nibrs["incident_date"].max().date()}')
print(f'Unique agencies     : {df_nibrs["agency_id"].nunique()}')
print()
print('Tier distribution:')
print(df_nibrs['Severity_Tier'].value_counts())

Loading NIBRS files...


NIBRS merged shape  : (605582, 11)
Date range          : 2024-01-01 -> 2024-12-31
Unique agencies     : 646

Tier distribution:
Severity_Tier
Tier 3 - Property             348293
Tier 2 - Personal Violence    148862
Tier 4 - Public Order          55805
Other                          45479
Tier 1 - Lethal                 7143
Name: count, dtype: int64


## Step 4: Build Daily Agency-Level Panel

In [4]:
df_nibrs['Date'] = df_nibrs['incident_date'].dt.normalize()

# Keep mapped tiers only
df_nibrs_eval = df_nibrs[df_nibrs['Severity_Tier'].isin(TIERS)].copy()

# Aggregate: count crimes per agency × day × tier
df_daily = (
    df_nibrs_eval
    .groupby(['agency_id', 'Date', 'Severity_Tier'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure all 4 tier columns exist
for t in TIERS:
    if t not in df_daily.columns:
        df_daily[t] = 0

df_daily = df_daily[['agency_id', 'Date'] + TIERS].copy()
df_daily = df_daily.sort_values(['agency_id', 'Date']).reset_index(drop=True)

print(f'Daily agency panel: {df_daily.shape}')
print(f'Unique agencies   : {df_daily["agency_id"].nunique()}')
print()
print('Class balance (positive rate):')
for i, t in enumerate(TIERS):
    rate = (df_daily[t] > 0).mean()
    print(f'  Tier {i+1}: {rate:.1%}')


Daily agency panel: (92729, 6)
Unique agencies   : 643

Class balance (positive rate):
  Tier 1: 5.1%
  Tier 2: 47.5%
  Tier 3: 77.5%
  Tier 4: 25.3%


## Step 5: Normalized Lag Features (Mean — same formula as Chicago)
Same computation as Chicago pipeline — rolling **mean** per agency.
This is scale-independent: a rural agency with 0.1 crimes/day and an urban agency
with 10 crimes/day both produce comparable lag_mean values relative to their own baseline.


In [5]:
print('Computing lag features (7d, 30d, 90d mean)...')

for i, tier in enumerate(TIERS):
    n   = i + 1
    grp = df_daily.groupby('agency_id')[tier]

    df_daily[f'tier{n}_lag_7d_mean']  = grp.transform(
        lambda x: x.shift(1).rolling(7,  min_periods=1).mean()
    )
    df_daily[f'tier{n}_lag_30d_mean'] = grp.transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean()
    )
    df_daily[f'tier{n}_lag_90d_mean'] = grp.transform(
        lambda x: x.shift(1).rolling(90, min_periods=1).mean()
    )
    print(f'  Tier {n} done.')

df_daily = df_daily.fillna(0)
print('Lag features complete.')


Computing lag features (7d, 30d, 90d mean)...


  Tier 1 done.


  Tier 2 done.


  Tier 3 done.


  Tier 4 done.
Lag features complete.


## Step 6: Surge Ratio, Trend & Cross-Tier Features
Exact same formulas as Chicago pipeline — these are scale-independent ratios
that generalize across jurisdictions of any size.


In [6]:
# Surge ratio: 7d_mean / 30d_mean
for i in range(1, 5):
    df_daily[f'tier{i}_surge_ratio'] = (
        df_daily[f'tier{i}_lag_7d_mean'] /
        df_daily[f'tier{i}_lag_30d_mean'].clip(lower=0.01)
    ).clip(upper=10.0)

# Trend: 7d_mean - 30d_mean
for i in range(1, 5):
    df_daily[f'tier{i}_trend'] = (
        df_daily[f'tier{i}_lag_7d_mean'] - df_daily[f'tier{i}_lag_30d_mean']
    )

# Cross-tier ratios
df_daily['violence_property_ratio'] = (
    df_daily['tier2_lag_7d_mean'] /
    df_daily['tier3_lag_7d_mean'].clip(lower=0.01)
).clip(upper=10.0)

df_daily['disorder_to_violence'] = (
    df_daily['tier4_lag_7d_mean'] /
    df_daily['tier2_lag_7d_mean'].clip(lower=0.01)
).clip(upper=10.0)

print('Surge ratios, trends, cross-tier features created.')


Surge ratios, trends, cross-tier features created.


## Step 7: Cyclical Time Features

In [7]:
df_daily['month_sin'] = np.sin(2 * np.pi * df_daily['Date'].dt.month     / 12)
df_daily['month_cos'] = np.cos(2 * np.pi * df_daily['Date'].dt.month     / 12)
df_daily['dow_sin']   = np.sin(2 * np.pi * df_daily['Date'].dt.dayofweek / 7)
df_daily['dow_cos']   = np.cos(2 * np.pi * df_daily['Date'].dt.dayofweek / 7)
print('Cyclical features created.')


Cyclical features created.


## Step 8: Composite Risk — Apply Chicago Weights
**IMPORTANT:** Use weights loaded from Chicago training (`composite_weights.joblib`).
Do **NOT** recompute correlations on NIBRS data — that would use NIBRS-specific patterns
and break the generalization test.


In [8]:
lag_7d_cols = [f'tier{i}_lag_7d_mean' for i in range(1, 5)]

for target in TARGET_COLS:
    n = target[-1]
    w = composite_weights[target]   # loaded from Chicago training
    df_daily[f'composite_risk_tier{n}'] = (
        df_daily['tier1_lag_7d_mean'] * w[0] +
        df_daily['tier2_lag_7d_mean'] * w[1] +
        df_daily['tier3_lag_7d_mean'] * w[2] +
        df_daily['tier4_lag_7d_mean'] * w[3]
    )

print('Composite risk features created using Chicago-trained weights:')
for target in TARGET_COLS:
    w = composite_weights[target]
    print(f'  {target}: T1={w[0]:.3f}, T2={w[1]:.3f}, T3={w[2]:.3f}, T4={w[3]:.3f}')


Composite risk features created using Chicago-trained weights:
  y_tier1: T1=0.364, T2=0.390, T3=0.054, T4=0.192
  y_tier2: T1=0.081, T2=0.560, T3=0.158, T4=0.201
  y_tier3: T1=0.045, T2=0.245, T3=0.410, T4=0.299
  y_tier4: T1=0.042, T2=0.188, T3=0.164, T4=0.607


## Step 9: Binary Targets

In [9]:
df_daily['y_tier1'] = (df_daily['Tier 1 - Lethal']           > 0).astype(int)
df_daily['y_tier2'] = (df_daily['Tier 2 - Personal Violence'] > 0).astype(int)
df_daily['y_tier3'] = (df_daily['Tier 3 - Property']          > 0).astype(int)
df_daily['y_tier4'] = (df_daily['Tier 4 - Public Order']      > 0).astype(int)

y_actual = df_daily[TARGET_COLS].copy()
print('Binary targets created:')
for col in TARGET_COLS:
    print(f'  {col}: {df_daily[col].mean():.1%} positive rate')


Binary targets created:
  y_tier1: 5.1% positive rate
  y_tier2: 47.5% positive rate
  y_tier3: 77.5% positive rate
  y_tier4: 25.3% positive rate


## Step 10: Align to 30 Features & Apply Chicago Scaler
1. Select exactly the same 30 features in the same order as Chicago training
2. Apply the **loaded scaler** (transform only — DO NOT refit)


In [10]:
# Ensure all 30 features exist (fill 0 if any missing)
for col in FEATURES:
    if col not in df_daily.columns:
        print(f'  WARNING: {col} missing — filling with 0')
        df_daily[col] = 0

X_nibrs        = df_daily[FEATURES].copy()
# Apply Chicago-fitted scaler — transform only, never refit
X_nibrs_scaled = loaded_scaler.transform(X_nibrs)

print(f'Feature matrix ready: {X_nibrs.shape}')
print(f'  Features aligned to Chicago training order: {len(FEATURES)}')
print(f'  Scaled matrix shape: {X_nibrs_scaled.shape}')
print()

# Sanity check: no nulls
null_count = X_nibrs.isnull().sum().sum()
print(f'  Null check: {null_count} nulls ({"PASSED" if null_count == 0 else "WARNING"})')


Feature matrix ready: (92729, 30)
  Features aligned to Chicago training order: 30
  Scaled matrix shape: (92729, 30)

  Null check: 0 nulls (PASSED)


## Step 11: Run All Models on NIBRS

In [11]:
all_results       = []
all_probabilities = {}

for model_name, estimators in loaded_models.items():
    print(f'\n--- Predicting with {model_name} ---')
    use_scaled = NEEDS_SCALING[model_name]
    X_input    = X_nibrs_scaled if use_scaled else X_nibrs.values

    prob_matrix = []
    pred_matrix = []

    for i, col in enumerate(TARGET_COLS):
        clf   = estimators[i]
        probs = clf.predict_proba(X_input)[:, 1]
        preds = (probs >= THRESHOLDS[i]).astype(int)
        prob_matrix.append(probs)
        pred_matrix.append(preds)

    prob_matrix = np.array(prob_matrix).T
    pred_matrix = np.array(pred_matrix).T
    all_probabilities[model_name] = prob_matrix

    for i, col in enumerate(TARGET_COLS):
        y_true = y_actual[col].values
        y_prob = prob_matrix[:, i]
        y_pred = pred_matrix[:, i]

        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = np.nan

        all_results.append({
            'Model'    : model_name,
            'Tier'     : col,
            'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
            'Recall'   : round(recall_score(y_true,    y_pred, zero_division=0), 4),
            'F1'       : round(f1_score(y_true,        y_pred, zero_division=0), 4),
            'AUC'      : round(auc, 4) if not np.isnan(auc) else np.nan
        })

results_df = pd.DataFrame(all_results)
print('\n=== NIBRS GENERALIZATION RESULTS (v2) ===')
print(results_df.to_string(index=False))



--- Predicting with Logistic Regression ---



--- Predicting with Random Forest ---



--- Predicting with XGBoost ---



--- Predicting with Neural Network ---



=== NIBRS GENERALIZATION RESULTS (v2) ===
              Model    Tier  Precision  Recall     F1    AUC
Logistic Regression y_tier1     0.0512  1.0000 0.0973 0.7400
Logistic Regression y_tier2     0.4757  0.9994 0.6446 0.7320
Logistic Regression y_tier3     0.7898  0.9761 0.8731 0.7250
Logistic Regression y_tier4     0.2892  0.9503 0.4434 0.7568
      Random Forest y_tier1     0.0512  0.9987 0.0975 0.6904
      Random Forest y_tier2     0.4762  0.9997 0.6451 0.7179
      Random Forest y_tier3     0.7896  0.9809 0.8749 0.7244
      Random Forest y_tier4     0.2837  0.9674 0.4387 0.7002
            XGBoost y_tier1     0.0333  0.0401 0.0364 0.5653
            XGBoost y_tier2     0.4873  0.9863 0.6523 0.7280
            XGBoost y_tier3     0.7822  0.9933 0.8752 0.7301
            XGBoost y_tier4     0.5019  0.3869 0.4370 0.7341
     Neural Network y_tier1     0.0369  0.0717 0.0487 0.3456
     Neural Network y_tier2     0.4882  0.9857 0.6530 0.7338
     Neural Network y_tier3     0.7824  0.

## Step 12: Compare v2 vs Original NIBRS Results

In [12]:
# Original NIBRS results from Phase 2 notebook
original = [
    ('Logistic Regression','y_tier1', 0.2211, 0.3931, 0.2830, 0.7329),
    ('Logistic Regression','y_tier2', 0.4764, 0.9991, 0.6452, 0.7222),
    ('Logistic Regression','y_tier3', 0.7795, 0.9965, 0.8748, 0.7166),
    ('Logistic Regression','y_tier4', 0.5617, 0.4531, 0.5016, 0.7483),
    ('Random Forest',       'y_tier1', 0.0419, 0.0162, 0.0234, 0.6134),
    ('Random Forest',       'y_tier2', 0.4894, 0.9697, 0.6505, 0.6176),
    ('Random Forest',       'y_tier3', 0.7818, 0.9809, 0.8701, 0.6760),
    ('Random Forest',       'y_tier4', 0.3904, 0.1380, 0.2039, 0.6090),
    ('XGBoost',             'y_tier1', 0.0327, 0.2711, 0.0583, 0.3244),
    ('XGBoost',             'y_tier2', 0.4600, 0.9345, 0.6166, 0.5606),
    ('XGBoost',             'y_tier3', 0.7769, 0.9980, 0.8737, 0.7023),
    ('XGBoost',             'y_tier4', 0.2347, 0.6969, 0.3511, 0.5272),
    ('Neural Network',      'y_tier1', 0.0816, 0.0034, 0.0065, 0.3939),
    ('Neural Network',      'y_tier2', 0.4833, 0.9878, 0.6490, 0.7218),
    ('Neural Network',      'y_tier3', 0.7801, 0.9957, 0.8749, 0.7237),
    ('Neural Network',      'y_tier4', 0.5602, 0.3382, 0.4218, 0.7447),
]
orig_df = pd.DataFrame(original, columns=['Model','Tier','Precision','Recall','F1','AUC'])

comp = results_df.merge(orig_df, on=['Model','Tier'], suffixes=('_v2','_orig'))
comp['F1_delta']  = (comp['F1_v2']  - comp['F1_orig']).round(4)
comp['AUC_delta'] = (comp['AUC_v2'] - comp['AUC_orig']).round(4)

print('=== F1 and AUC: v2 vs Original NIBRS Results ===')
print(comp[['Model','Tier','F1_orig','F1_v2','F1_delta','AUC_orig','AUC_v2','AUC_delta']].to_string(index=False))
print()
print(f'Mean F1  improvement: {comp["F1_delta"].mean():+.4f}')
print(f'Mean AUC improvement: {comp["AUC_delta"].mean():+.4f}')

# Highlight XGBoost specifically (was the worst generalizer)
xgb_comp = comp[comp['Model'] == 'XGBoost']
print(f'\nXGBoost AUC improvement (was worst generalizer):')
for _, row in xgb_comp.iterrows():
    print(f'  {row["Tier"]}: {row["AUC_orig"]:.4f} -> {row["AUC_v2"]:.4f}  ({row["AUC_delta"]:+.4f})')


=== F1 and AUC: v2 vs Original NIBRS Results ===
              Model    Tier  F1_orig  F1_v2  F1_delta  AUC_orig  AUC_v2  AUC_delta
Logistic Regression y_tier1   0.2830 0.0973   -0.1857    0.7329  0.7400     0.0071
Logistic Regression y_tier2   0.6452 0.6446   -0.0006    0.7222  0.7320     0.0098
Logistic Regression y_tier3   0.8748 0.8731   -0.0017    0.7166  0.7250     0.0084
Logistic Regression y_tier4   0.5016 0.4434   -0.0582    0.7483  0.7568     0.0085
      Random Forest y_tier1   0.0234 0.0975    0.0741    0.6134  0.6904     0.0770
      Random Forest y_tier2   0.6505 0.6451   -0.0054    0.6176  0.7179     0.1003
      Random Forest y_tier3   0.8701 0.8749    0.0048    0.6760  0.7244     0.0484
      Random Forest y_tier4   0.2039 0.4387    0.2348    0.6090  0.7002     0.0912
            XGBoost y_tier1   0.0583 0.0364   -0.0219    0.3244  0.5653     0.2409
            XGBoost y_tier2   0.6166 0.6523    0.0357    0.5606  0.7280     0.1674
            XGBoost y_tier3   0.8737 0

## Step 13: Jurisdictional Evaluation — PAI (Tier 2)

In [13]:
spatial_results = []
total_days      = len(df_daily)
total_tier2     = df_daily['y_tier2'].sum()
base_rate       = total_tier2 / total_days

for model_name, prob_matrix in all_probabilities.items():
    preds  = (prob_matrix[:, 1] >= THRESHOLDS[1]).astype(int)
    hits   = ((preds == 1) & (df_daily['y_tier2'].values == 1)).sum()
    flags  = preds.sum()

    sp_prec   = hits / flags         if flags > 0         else 0
    lift      = sp_prec / base_rate  if base_rate > 0     else 0
    hit_rate  = hits / total_tier2   if total_tier2 > 0   else 0
    flag_rate = flags / total_days   if total_days > 0    else 0
    pai       = hit_rate / flag_rate if flag_rate > 0     else 0

    spatial_results.append({
        'Model'            : model_name,
        'Base Rate'        : round(base_rate, 4),
        'Spatial Precision': round(sp_prec, 4),
        'Lift'             : round(lift, 4),
        'PAI'              : round(pai, 4),
        'Days Flagged'     : int(flags)
    })

spatial_df = pd.DataFrame(spatial_results)
print('=== JURISDICTIONAL EVALUATION — TIER 2 (PAI) ===')
print(spatial_df.to_string(index=False))
print()
print('PAI > 1.0 = model flags better-than-random jurisdictions')
print('PAI = 2.0 = flagged areas are 2x more likely to have crime than average')


=== JURISDICTIONAL EVALUATION — TIER 2 (PAI) ===
              Model  Base Rate  Spatial Precision   Lift    PAI  Days Flagged
Logistic Regression     0.4754             0.4757 1.0007 1.0007         92611
      Random Forest     0.4754             0.4762 1.0018 1.0018         92534
            XGBoost     0.4754             0.4873 1.0251 1.0251         89220
     Neural Network     0.4754             0.4882 1.0269 1.0269         89011

PAI > 1.0 = model flags better-than-random jurisdictions
PAI = 2.0 = flagged areas are 2x more likely to have crime than average


## Step 14: Threshold Sensitivity — Tier 2

In [14]:
threshold_grid = [0.20, 0.25, 0.35, 0.45, 0.55, 0.65]
y_true_t2      = df_daily['y_tier2'].values
base_rate_t2   = y_true_t2.mean()

print(f'Base rate Tier 2: {base_rate_t2:.3f}')
print()

for model_name, prob_matrix in all_probabilities.items():
    print(f'--- {model_name} ---')
    print(f'  {"Threshold":<10} {"Precision":<12} {"Recall":<10} {"F1":<8} {"Lift":<8} {"Days Flagged"}')
    print('  ' + '-' * 62)
    for t in threshold_grid:
        pred  = (prob_matrix[:, 1] >= t).astype(int)
        flags = pred.sum()
        if flags > 0:
            prec = precision_score(y_true_t2, pred, zero_division=0)
            rec  = recall_score(y_true_t2,    pred, zero_division=0)
            f1   = f1_score(y_true_t2,        pred, zero_division=0)
            lift = prec / base_rate_t2
        else:
            prec = rec = f1 = lift = 0
        print(f'  {t:<10.2f} {prec:<12.1%} {rec:<10.1%} {f1:<8.4f} {lift:<8.2f}x {flags}')
    print()


Base rate Tier 2: 0.475

--- Logistic Regression ---
  Threshold  Precision    Recall     F1       Lift     Days Flagged
  --------------------------------------------------------------
  0.20       47.6%        99.9%      0.6446   1.00    x 92611
  0.25       48.0%        99.3%      0.6475   1.01    x 91100
  0.35       51.1%        94.8%      0.6638   1.07    x 81784
  0.45       57.4%        81.5%      0.6735   1.21    x 62659
  0.55       65.5%        64.5%      0.6503   1.38    x 43431
  0.65       72.9%        49.6%      0.5908   1.53    x 30012

--- Random Forest ---
  Threshold  Precision    Recall     F1       Lift     Days Flagged
  --------------------------------------------------------------
  0.20       47.6%        100.0%     0.6451   1.00    x 92534


  0.25       48.0%        99.5%      0.6476   1.01    x 91418


  0.35       50.2%        96.4%      0.6599   1.06    x 84725


  0.45       55.4%        86.0%      0.6739   1.17    x 68363
  0.55       64.6%        66.6%      0.6557   1.36    x 45443
  0.65       74.6%        35.1%      0.4775   1.57    x 20732

--- XGBoost ---
  Threshold  Precision    Recall     F1       Lift     Days Flagged
  --------------------------------------------------------------
  0.20       48.7%        98.6%      0.6523   1.03    x 89220
  0.25       50.2%        96.5%      0.6604   1.06    x 84802
  0.35       54.7%        87.9%      0.6744   1.15    x 70887
  0.45       63.2%        68.9%      0.6592   1.33    x 48067
  0.55       74.8%        46.3%      0.5722   1.57    x 27307
  0.65       82.4%        22.7%      0.3559   1.73    x 12152

--- Neural Network ---
  Threshold  Precision    Recall     F1       Lift     Days Flagged
  --------------------------------------------------------------
  0.20       48.8%        98.6%      0.6530   1.03    x 89011
  0.25       50.2%        96.5%      0.6608   1.06    x 84707


  0.35       54.9%        87.0%      0.6736   1.16    x 69828


  0.45       62.6%        70.6%      0.6637   1.32    x 49662
  0.55       73.2%        49.4%      0.5904   1.54    x 29765
  0.65       82.5%        33.5%      0.4767   1.73    x 17916



## Step 15: Weekday vs Weekend Accuracy (Tier 2)

In [15]:
df_daily['is_weekend'] = df_daily['Date'].dt.dayofweek >= 5
weekend_mask = df_daily['is_weekend'].values
weekday_mask = ~weekend_mask
y_t2         = df_daily['y_tier2'].values

print(f'  {"Model":<22} {"Weekday Acc":>12} {"Weekend Acc":>12} {"Diff":>8}')
print('  ' + '-' * 58)
for model_name, prob_matrix in all_probabilities.items():
    pred   = (prob_matrix[:, 1] >= 0.20).astype(int)
    wd_acc = (pred[weekday_mask] == y_t2[weekday_mask]).mean()
    we_acc = (pred[weekend_mask] == y_t2[weekend_mask]).mean()
    print(f'  {model_name:<22} {wd_acc:>11.2%} {we_acc:>11.2%} {we_acc-wd_acc:>+7.2%}')


  Model                   Weekday Acc  Weekend Acc     Diff
  ----------------------------------------------------------
  Logistic Regression         45.70%      52.62%  +6.92%
  Random Forest               45.84%      52.64%  +6.80%
  XGBoost                     48.63%      53.67%  +5.04%
  Neural Network              48.85%      53.72%  +4.87%


## Step 16: Final Summary

In [16]:
print('=' * 70)
print('  NIBRS GENERALIZATION TEST v2 — COMPLETE SUMMARY')
print('=' * 70)
print(f'  NIBRS records     : {len(df_nibrs):,}')
print(f'  Agency-day rows   : {len(df_daily):,}')
print(f'  Unique agencies   : {df_daily["agency_id"].nunique()}')
print(f'  Features used     : {len(FEATURES)} (same 30 as Chicago training)')
print()
print('  Generalization Performance (AUC — v2):')
pivot = results_df.pivot(index='Model', columns='Tier', values='AUC')
print(pivot.round(4).to_string())
print()
print('  Generalization Performance (F1 — v2):')
pivot_f1 = results_df.pivot(index='Model', columns='Tier', values='F1')
print(pivot_f1.round(4).to_string())
print()
print('  Improvement vs original NIBRS testing:')
print(f'    Mean F1  change : {comp["F1_delta"].mean():+.4f}')
print(f'    Mean AUC change : {comp["AUC_delta"].mean():+.4f}')
print()
print('  PAI Summary (Tier 2):')
for _, row in spatial_df.iterrows():
    flag = " ← best" if row['PAI'] == spatial_df['PAI'].max() else ""
    print(f'    {row["Model"]:<22}: PAI={row["PAI"]:.4f}, Lift={row["Lift"]:.4f}x{flag}')


  NIBRS GENERALIZATION TEST v2 — COMPLETE SUMMARY
  NIBRS records     : 605,582
  Agency-day rows   : 92,729
  Unique agencies   : 643
  Features used     : 30 (same 30 as Chicago training)

  Generalization Performance (AUC — v2):
Tier                 y_tier1  y_tier2  y_tier3  y_tier4
Model                                                  
Logistic Regression   0.7400   0.7320   0.7250   0.7568
Neural Network        0.3456   0.7338   0.7261   0.7593
Random Forest         0.6904   0.7179   0.7244   0.7002
XGBoost               0.5653   0.7280   0.7301   0.7341

  Generalization Performance (F1 — v2):
Tier                 y_tier1  y_tier2  y_tier3  y_tier4
Model                                                  
Logistic Regression   0.0973   0.6446   0.8731   0.4434
Neural Network        0.0487   0.6530   0.8754   0.4899
Random Forest         0.0975   0.6451   0.8749   0.4387
XGBoost               0.0364   0.6523   0.8752   0.4370

  Improvement vs original NIBRS testing:
    Mean F1  